In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1110, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 1110 (delta 97), reused 0 (delta 0), pack-reused 961 (from 2)
Receiving objects: 100% (1110/1110), 513.98 MiB | 43.52 MiB/s, done.
Resolving deltas: 100% (617/617), done.
Updating files: 100% (210/210), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.4 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorization_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorization_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
Matrix

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'knn_study.db'
json_name_item = 'item_knn_best_params.json'
json_name_user = 'user_knn_best_params.json'

config = {
    'tune_parameters_item_knn': True,
    'tune_parameters_user_knn': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name_item': json_name_item,
    'json_name_user': json_name_user,
    'copy_prev_tuning_db': True,
    'train_best_model': False,
    'copy_prev_best_params': True,
    'save_github': True,
    'candidates': '50',
    'n_trials_item': 1000, # 1000 -> 5h
    'n_trials_user': 300  # 300 -> 5h
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'item_knn_Optimization')
(2, 'hyperparameters_tuning_knn_item')
(3, 'user_knn_Optimization')
(4, 'item_knn_fine_tuning_Optimization')
(5, 'user_knn_fine_tuning_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_itemknn(optuna_trial):
    recommender_instance = ItemKNNCFRecommender(URM_train)
    
    # 1. Choose the similarity metric first
    similarity = optuna_trial.suggest_categorical("similarity", 
        ['cosine', 'asymmetric', 'dice', 'jaccard', 'tanimoto', 'tversky', 'euclidean'])
    
    # 2. Global parameters
    params = {
        'similarity': similarity,
        'topK': optuna_trial.suggest_int("topK", 10, 1000),
        'shrink': optuna_trial.suggest_int("shrink", 0, 500),
        'normalize': optuna_trial.suggest_categorical("normalize", [True, False]),
        'feature_weighting': optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"]),
    }
    
    # 3. Conditional parameters based on similarity
    if similarity == 'asymmetric':
        params["asymmetric_alpha"] = optuna_trial.suggest_float("asymmetric_alpha", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'tversky':
        params['tversky_alpha'] = optuna_trial.suggest_float("tversky_alpha", 0, 2.0, log=False)
        params['tversky_beta'] = optuna_trial.suggest_float("tversky_beta", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'euclidean':
        params["normalize_avg_row"] = optuna_trial.suggest_categorical("normalize_avg_row", [True, False])
        params["similarity_from_distance_mode"] = optuna_trial.suggest_categorical("similarity_from_distance_mode", ["lin", "log", "exp"])
        params["normalize"] = optuna_trial.suggest_categorical("normalize", [True, False])

    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time
# an Itemknn study_name MUST contain "item_knn"
if config['tune_parameters_item_knn']:
    study = optuna.create_study(direction='maximize', study_name='item_knn_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_itemknn, n_trials=config['n_trials_item'])

[I 2025-12-26 00:08:57,782] Using an existing study with name 'item_knn_Optimization' instead of creating a new one.


Previous trials found: 1805
Similarity column 6969 (100.0%), 1827.15 column/sec. Elapsed time 3.81 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 17.07 sec. Users per second: 1586


[I 2025-12-26 00:09:20,499] Trial 1805 finished with value: 0.3960423602482102 and parameters: {'similarity': 'asymmetric', 'topK': 28, 'shrink': 294, 'normalize': True, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.16746480930852625}. Best is trial 491 with value: 0.39735827200668467.


Similarity column 6969 (100.0%), 1790.09 column/sec. Elapsed time 3.89 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 19.25 sec. Users per second: 1406


[I 2025-12-26 00:09:44,216] Trial 1806 finished with value: 0.36705689944170505 and parameters: {'similarity': 'asymmetric', 'topK': 66, 'shrink': 436, 'normalize': True, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.33196958718734093}. Best is trial 491 with value: 0.39735827200668467.


...

Similarity column 6969 (100.0%), 1836.67 column/sec. Elapsed time 3.79 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 18.47 sec. Users per second: 1465


[I 2025-12-26 06:35:21,649] Trial 2803 finished with value: 0.350471239590761 and parameters: {'similarity': 'tversky', 'topK': 71, 'shrink': 411, 'normalize': False, 'feature_weighting': 'TF-IDF', 'tversky_alpha': 0.59673118388952, 'tversky_beta': 1.6774531910674708}. Best is trial 491 with value: 0.39735827200668467.


Similarity column 6969 (100.0%), 1869.01 column/sec. Elapsed time 3.73 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 16.71 sec. Users per second: 1620


[I 2025-12-26 06:35:42,657] Trial 2804 finished with value: 0.39310213463304067 and parameters: {'similarity': 'asymmetric', 'topK': 28, 'shrink': 423, 'normalize': True, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.24188722504524598}. Best is trial 491 with value: 0.39735827200668467.


CPU times: user 6h 19min 8s, sys: 7min 41s, total: 6h 26min 49s
Wall time: 6h 26min 46s


In [17]:
if config['save_json'] and config['save_github'] and config['tune_parameters_item_knn']:
    j_name = config['json_name_item']
    l_path = f"/kaggle/working/{j_name}"
    r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
    
    best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
    
    try:
        for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
            if "item_knn" not in s.study_name: 
                continue

            if s.best_trial and s.best_trial.value > best_val:
                best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
    except:
        best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

    prev_val, exists = -float('inf'), False
    try:
        prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
        exists = True
    except:
        pass

    if not exists or best_val > prev_val:
        print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
        with open(l_path, 'w') as f:
            json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
        if config.get('save_github'):
            upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
    else:
        print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.398834 <= Saved: 0.398834


In [18]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_userknn(optuna_trial):
    recommender_instance = UserKNNCFRecommender(URM_train)
    
    # 1. Choose the similarity metric first
    similarity = optuna_trial.suggest_categorical("similarity", 
        ['cosine', 'asymmetric', 'dice', 'jaccard', 'tanimoto', 'tversky', 'euclidean'])
    
    # 2. Global parameters
    params = {
        'similarity': similarity,
        'topK': optuna_trial.suggest_int("topK", 10, 1000),
        'shrink': optuna_trial.suggest_int("shrink", 0, 500),
        'normalize': optuna_trial.suggest_categorical("normalize", [True, False]),
        'feature_weighting': optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"]),
    }
    
    # 3. Conditional parameters based on similarity
    if similarity == 'asymmetric':
        params["asymmetric_alpha"] = optuna_trial.suggest_float("asymmetric_alpha", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'tversky':
        params['tversky_alpha'] = optuna_trial.suggest_float("tversky_alpha", 0, 2.0, log=False)
        params['tversky_beta'] = optuna_trial.suggest_float("tversky_beta", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'euclidean':
        params["normalize_avg_row"] = optuna_trial.suggest_categorical("normalize_avg_row", [True, False])
        params["similarity_from_distance_mode"] = optuna_trial.suggest_categorical("similarity_from_distance_mode", ["lin", "log", "exp"])
        params["normalize"] = optuna_trial.suggest_categorical("normalize", [True, False])

    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [19]:
%%time
# an Userknn study_name MUST contain "user_knn"
if config['tune_parameters_user_knn']:
    study = optuna.create_study(direction='maximize', study_name='user_knn_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_userknn, n_trials=config['n_trials_user'])

[I 2025-12-26 06:35:46,784] Using an existing study with name 'user_knn_Optimization' instead of creating a new one.


Previous trials found: 905
Similarity column 27095 (100.0%), 703.78 column/sec. Elapsed time 38.50 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 24.54 sec. Users per second: 1103


[I 2025-12-26 06:36:51,317] Trial 905 finished with value: 0.3663862106382509 and parameters: {'similarity': 'asymmetric', 'topK': 363, 'shrink': 165, 'normalize': False, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.3092189978605275}. Best is trial 437 with value: 0.367006989194398.


Similarity column 27095 (100.0%), 698.26 column/sec. Elapsed time 38.80 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 25.25 sec. Users per second: 1072


[I 2025-12-26 06:37:56,374] Trial 906 finished with value: 0.36662053442171094 and parameters: {'similarity': 'asymmetric', 'topK': 405, 'shrink': 213, 'normalize': False, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.22441511459960725}. Best is trial 437 with value: 0.367006989194398.


...

Similarity column 27095 (100.0%), 718.94 column/sec. Elapsed time 37.69 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 23.00 sec. Users per second: 1177


[I 2025-12-26 12:02:34,897] Trial 1203 finished with value: 0.29599840575643405 and parameters: {'similarity': 'jaccard', 'topK': 374, 'shrink': 225, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 437 with value: 0.367006989194398.


Similarity column 27095 (100.0%), 704.46 column/sec. Elapsed time 38.46 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 24.37 sec. Users per second: 1111


[I 2025-12-26 12:03:38,744] Trial 1204 finished with value: 0.3665438430758976 and parameters: {'similarity': 'asymmetric', 'topK': 350, 'shrink': 273, 'normalize': False, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.19597833634195078}. Best is trial 437 with value: 0.367006989194398.


CPU times: user 5h 23min 20s, sys: 3min 46s, total: 5h 27min 6s
Wall time: 5h 27min 51s


In [20]:
if config['save_json'] and config['save_github'] and config['tune_parameters_user_knn']:
    j_name = config['json_name_user']
    l_path = f"/kaggle/working/{j_name}"
    r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
    
    best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
    
    try:
        for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
            if "user_knn" not in s.study_name: 
                continue

            if s.best_trial and s.best_trial.value > best_val:
                best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
    except:
        best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

    prev_val, exists = -float('inf'), False
    try:
        prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
        exists = True
    except:
        pass

    if not exists or best_val > prev_val:
        print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
        with open(l_path, 'w') as f:
            json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
        if config.get('save_github'):
            upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
    else:
        print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.367162 <= Saved: 0.367162


In [21]:
if config['save_github'] and (config['tune_parameters_user_knn'] or config['tune_parameters_item_knn']):
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )